# Virtual-trial divergence — the headline feature

**NOT FOR CLINICAL USE.** Population/trial-level forward simulation only. This notebook is executed in CI (nbmake) so the figure cannot silently break.

It overlays every *eligible* TGI model for a chosen tumor context, greys out the models whose transportability envelope the context violates, and quantifies how much the population OS prediction depends on the model choice.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import onkos

ds = onkos.load()
print(f'{len(ds)} records | tiers {dict(ds.by_tier())}')

In [ ]:
ctx = {'tumor_type': 'NSCLC', 'line': 'first'}
t = np.linspace(0, 104, 209)
cmp = onkos.compare(ds, purpose='tgi', context=ctx, drug_effect=1.0, t=t)

print('included:', [tr.record_id for tr in cmp.included])
for rid, reason in cmp.excluded:
    print('excluded:', rid, '->', reason)
print(f'OS divergence (max pointwise): {cmp.os_divergence:.3f}')
print('median OS range (weeks):', cmp.median_os_range)
assert cmp.os_divergence > 0  # model choice matters

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
for tr in cmp.included:
    ax1.plot(t, tr.tumor_size, label=f'{tr.record_id} [{tr.tier}]')
    ax2.plot(t, tr.os_curve, label=tr.record_id)
ax1.set(title='Tumor size', xlabel='weeks', ylabel='SLD (mm)'); ax1.legend(fontsize=7)
ax2.axhline(0.5, ls=':', color='grey'); ax2.set(title='Population OS', xlabel='weeks', ylabel='S(t)')
ax2.legend(fontsize=7)
fig.tight_layout()

In [ ]:
# Tier propagation: applying an NSCLC model out of context floors the result to D.
oc = onkos.simulate(ds, 'resistance.claret_2009.tgi', context={'tumor_type': 'colorectal', 'line': 'first'})
print('out-of-context tier:', oc.tier)
assert oc.tier == 'D'
for w in oc.warnings:
    print(' !', w)